## Data Cleaning and Validation
## Objective
The goal of this notebook is to load the Walmart sales dataset, verify its structure, 
check for missing values, duplicates, and timestamp consistency, and ensure the data 
is suitable for time series forecasting.


In [25]:
import pandas as pd

train = pd.read_csv("../data/raw/train.csv")
features = pd.read_csv("../data/raw/features.csv")
stores = pd.read_csv("../data/raw/stores.csv")



In [26]:
train.head()
train.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Store         421570 non-null  int64  
 1   Dept          421570 non-null  int64  
 2   Date          421570 non-null  object 
 3   Weekly_Sales  421570 non-null  float64
 4   IsHoliday     421570 non-null  bool   
dtypes: bool(1), float64(1), int64(2), object(1)
memory usage: 13.3+ MB


- The dataset contains weekly sales data per store and department.
- The date column is currently in string format and will be converted to datetime.


Converting date column to datetime

In [27]:
train["Date"] = pd.to_datetime(train["Date"])
features["Date"] = pd.to_datetime(features["Date"])

Time series models require a properly formatted datetime index.


#### Checking for missing values

In [28]:
train.isnull().sum()

Store           0
Dept            0
Date            0
Weekly_Sales    0
IsHoliday       0
dtype: int64

- No missing values were found in the core sales data.
- Missing values in auxiliary datasets (features) will be handled during feature engineering if used.


### Checking for duplicates

In [29]:
train.duplicated().sum()

np.int64(0)

- No duplicate records were detected in the dataset

### Check time frequency consistency

In [30]:
train.sort_values("Date" , inplace = True)

train["Date"].diff().value_counts().head()

Date
0 days    421427
7 days       142
Name: count, dtype: int64

- The dataset follows a consistent weekly frequency.
- No missing timestamps were detected.


Outlier santiy check (light, not aggresive)

In [31]:
train["Weekly_Sales"].describe()


count    421570.000000
mean      15981.258123
std       22711.183519
min       -4988.940000
25%        2079.650000
50%        7612.030000
75%       20205.852500
max      693099.360000
Name: Weekly_Sales, dtype: float64

- Sales values show natural variation across stores and departments.
- Extreme values are retained as they likely represent real-world demand spikes.


Merge dataset

Although multiple auxiliary features are available, only time-consistent features are retained for forecasting to avoid data leakage

In [32]:
df = train.merge(features, on=["Store", "Date"], how="left")
df = df.merge(stores, on="Store", how="left")


Before merging time series datasets, I ensured consistent datetime formats to avoid type mismatch errors during joins.

In [33]:
print(train.dtypes)
print(features.dtypes)

Store                    int64
Dept                     int64
Date            datetime64[ns]
Weekly_Sales           float64
IsHoliday                 bool
dtype: object
Store                    int64
Date            datetime64[ns]
Temperature            float64
Fuel_Price             float64
MarkDown1              float64
MarkDown2              float64
MarkDown3              float64
MarkDown4              float64
MarkDown5              float64
CPI                    float64
Unemployment           float64
IsHoliday                 bool
dtype: object


- The Date column was converted to datetime format across all datasets to ensure consistency during time-based merges.

- Sort by date

In [34]:
df = df.sort_values("Date")


### Saving cleaned dataset

In [35]:
df.to_csv("../data/cleaned/walmart_sales_cleaned.csv", index=False)

In [36]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/cleaned/walmart_sales_cleaned.csv")

# Critical fixes
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Store", "Dept", "Date"])

# Holiday column conflict → trust the train file version
df = df.drop(columns=["IsHoliday_y"]).rename(columns={"IsHoliday_x": "IsHoliday"})

# Useful: total markdown spend
df["MarkdownTotal"] = df[[f"MarkDown{i}" for i in range(1,6)]].sum(axis=1, min_count=1)
df["MarkdownAny"]   = df["MarkdownTotal"].notna()

print(df.isna().mean().sort_values(ascending=False).head(12))   # check missingness

MarkDown2        0.736110
MarkDown4        0.679847
MarkDown3        0.674808
MarkDown1        0.642572
MarkDown5        0.640790
MarkdownTotal    0.640790
Dept             0.000000
Store            0.000000
Temperature      0.000000
Date             0.000000
Fuel_Price       0.000000
Weekly_Sales     0.000000
dtype: float64


Conclusion:
The dataset contains consistent weekly timestamps, no missing values, and no major anomalies. Therefore, no aggressive cleaning or resampling was required.